<p><font size="6" color='grey'> <b>
KI-Agenten. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>



<p><font size="5" color='grey'> <b>
Multi-Agent Projekt
</b></font> </br></p>

---

In [ ]:
#@title 🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

import os
os.environ["LANGSMITH_TRACING"]  = "true"
os.environ["LANGSMITH_PROJECT"]  = "M19-Multi-Agent-Projekt"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment, get_ipinfo, setup_api_keys,
    mprint, install_packages, mermaid, load_prompt
)
setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

In [ ]:
#@title 📦 Pakete installieren{ display-mode: "form" }
install_packages([
    'wikipedia',
])

# 1 | Übersicht
---

Die Module **M17** und **M18** haben das Supervisor Pattern erarbeitet.
**M19** bringt alles zusammen: Wir bauen ein eigenes Projekt von Grund auf.

| Thema | M17 | M18 | M19 |
|-------|-----|-----|-----|
| Supervisor-Pattern | ✅ Deep Dive | ✅ Vertiefung | ✅ Anwendung |
| Worker mit Tools | ✅ 3 Agents | ✅ Resilient | ✅ **Projekt-spezifisch** |
| Iterations-Schutz | ✅ | ✅ Retry | ✅ |
| Prompt-Management | Inline | Inline | ✅ **load_prompt()** |
| Projekt-Templates | ❌ | ❌ | ✅ **3 Templates** |
| MVP-Checkliste | ❌ | ❌ | ✅ |

**Lernziele:**
- Drei wiederverwendbare Projekt-Templates verstehen und anwenden
- Das passende Template für eigene Aufgaben auswählen
- Supervisor-Prompt mit `load_prompt()` aus Datei laden
- Ein vollständiges Projekt selbstständig implementieren
- LangSmith-Traces gezielt für Debugging nutzen

In [ ]:
#@markdown   <p><font size="4" color='green'>  flowchart: Kurs-Fortschritt M12 → M19</font> </br></p>

diagram = '''
%%{init: {'theme':'dark'}}%%
flowchart LR
    M12["M12 StateGraph"]
    M14["M14 Routing"]
    M15["M15 LangSmith"]
    M17["M17 Human in Loop"]
    M18["M18 Multi-Agent\nPatterns"]
    M19_sup["M19 Supervisor\nDeep Dive"]
    M19(["🎯 M19\nProjekt"])

    M12 --> M14 --> M15 --> M17 --> M18 --> M19_sup --> M19

    style M12     fill:#37474F,color:#fff
    style M14     fill:#37474F,color:#fff
    style M15     fill:#37474F,color:#fff
    style M17     fill:#37474F,color:#fff
    style M18     fill:#37474F,color:#fff
    style M19_sup fill:#37474F,color:#fff
    style M19     fill:#FF9800,color:#000
'''
mermaid(diagram, width=1000)

# 2 | Projekt-Templates
---

Drei bewährte Templates decken die häufigsten Multi-Agent-Anwendungsfälle ab.
Jedes Template ist ein vollständiger Supervisor-Graph – direkt einsetzbar.

<p><font color='black' size="5">Template A – Content Pipeline</font></p>

**Einsatz:** Artikel, Berichte, Zusammenfassungen, Blogposts  
**Agents:** Recherche → Schreiben  
**Kernidee:** Fakten zuerst sammeln, dann strukturiert verfassen.


In [ ]:
#@markdown   <p><font size="4" color='green'>  flowchart: Template A – Content Pipeline</font> </br></p>

diag_a = '''
%%{init: {'theme':'dark'}}%%
flowchart TD
    START([START]) --> SUP["🎯 Supervisor\nContent-Koordinator"]

    SUP -->|"recherche"| R["🔍 Recherche-Agent\nFakten sammeln"]
    SUP -->|"schreiben"| W["✍️ Schreib-Agent\nText erstellen"]
    SUP -->|"FINISH"| END([END])

    R -->|"AIMessage\n(name=Recherche)"| SUP
    W -->|"AIMessage\n(name=Schreiben)"| SUP

    R --- RT1["📚 wikipedia_suche"]
    W --- WT1["📋 gliederung_erstellen"]
    W --- WT2["🔢 wort_zaehlen"]

    style SUP fill:#FF9800,color:#000
    style R   fill:#2196F3,color:#fff
    style W   fill:#4CAF50,color:#fff
    style RT1 fill:#1565C0,color:#fff
    style WT1 fill:#2E7D32,color:#fff
    style WT2 fill:#2E7D32,color:#fff
'''
mermaid(diag_a, width=700)


<p><font color='black' size="5">Template B – Analyse Pipeline</font></p>

**Einsatz:** Datenanalyse, Code-Reviews, Auswertungen  
**Agents:** Erfassen → Analysieren  
**Kernidee:** Rohdaten aufnehmen, dann auswerten und interpretieren.


In [ ]:
#@markdown   <p><font size="4" color='green'>  flowchart: Template B – Analyse Pipeline</font> </br></p>

diag_b = '''
%%{init: {'theme':'dark'}}%%
flowchart TD
    START([START]) --> SUP["🎯 Supervisor\nAnalyse-Koordinator"]

    SUP -->|"erfassen"| E["📥 Erfassungs-Agent\nDaten aufnehmen"]
    SUP -->|"analysieren"| A["🔬 Analyse-Agent\nAuswerten"]
    SUP -->|"FINISH"| END([END])

    E -->|"AIMessage\n(name=Erfassen)"| SUP
    A -->|"AIMessage\n(name=Analyse)"| SUP

    E --- ET1["📊 daten_laden"]
    A --- AT1["▶️ python_ausfuehren"]
    A --- AT2["🔍 muster_suchen"]

    style SUP fill:#FF9800,color:#000
    style E   fill:#9C27B0,color:#fff
    style A   fill:#F44336,color:#fff
    style ET1 fill:#6A1B9A,color:#fff
    style AT1 fill:#B71C1C,color:#fff
    style AT2 fill:#B71C1C,color:#fff
'''
mermaid(diag_b, width=700)


<p><font color='black' size="5">Template C – Support Pipeline</font></p>

**Einsatz:** Ticket-Bearbeitung, FAQ-Systeme, Kundenservice  
**Agents:** Klassifizieren → Lösen  
**Kernidee:** Anfrage kategorisieren, dann zielgerichtet beantworten.

In [ ]:
#@markdown   <p><font size="4" color='green'>  flowchart: Template C – Support Pipeline</font> </br></p>

diag_c = '''
%%{init: {'theme':'dark'}}%%
flowchart TD
    START([START]) --> SUP["🎯 Supervisor\nSupport-Koordinator"]

    SUP -->|"klassifizieren"| K["🏷️ Klassifizier-Agent\nKategorie bestimmen"]
    SUP -->|"loesen"| L["🔧 Lösungs-Agent\nAntwort generieren"]
    SUP -->|"FINISH"| END([END])

    K -->|"AIMessage\n(name=Klassifizierung)"| SUP
    L -->|"AIMessage\n(name=Loesung)"| SUP

    K --- KT1["🗂️ ticket_kategorisieren"]
    L --- LT1["📖 faq_suchen"]
    L --- LT2["✅ antwort_validieren"]

    style SUP fill:#FF9800,color:#000
    style K   fill:#00BCD4,color:#000
    style L   fill:#8BC34A,color:#000
    style KT1 fill:#006064,color:#fff
    style LT1 fill:#33691E,color:#fff
    style LT2 fill:#33691E,color:#fff
'''
mermaid(diag_c, width=700)

**Template-Vergleich:**

| | Template A | Template B | Template C |
|---|---|---|---|
| **Name** | Content Pipeline | Analyse Pipeline | Support Pipeline |
| **Agents** | Recherche, Schreiben | Erfassen, Analysieren | Klassifizieren, Lösen |
| **Input** | Thema / Frage | Daten / Code | Anfrage / Ticket |
| **Output** | Strukturierter Text | Auswertung / Bericht | Kategorisierte Antwort |
| **Typisches Tool** | wikipedia_suche | python_ausfuehren | faq_suchen |
| **Komplexität** | ⭐⭐ | ⭐⭐⭐ | ⭐⭐ |

# 3 | Template wählen
---

Die Wahl des richtigen Templates hängt vom **Output-Typ** ab:

- **Text/Inhalt erzeugen** → Template A (Content)
- **Daten/Code auswerten** → Template B (Analyse)
- **Anfragen bearbeiten** → Template C (Support)

Nutze den Entscheidungsbaum als erste Orientierung:

In [ ]:
#@markdown   <p><font size="4" color='green'>  flowchart: Template-Entscheidungsbaum</font> </br></p>

diag_e = '''
%%{init: {'theme':'dark'}}%%
flowchart TD
    START(["❓ Mein Anwendungsfall"]) --> Q1{"Geht es um\nText-Erstellung?"}

    Q1 -->|"JA"| Q2{"Brauche ich\nFakten-Recherche?"}
    Q1 -->|"NEIN"| Q3{"Geht es um\nDatenauswertung?"}

    Q2 -->|"JA"| A["✅ Template A\nContent Pipeline"]
    Q2 -->|"NEIN"| A2["✅ Template A\n(nur Schreib-Agent)"]

    Q3 -->|"JA"| B["✅ Template B\nAnalyse Pipeline"]
    Q3 -->|"NEIN"| Q4{"Bearbeite ich\nAnfragen?"}

    Q4 -->|"JA"| C["✅ Template C\nSupport Pipeline"]
    Q4 -->|"NEIN"| CUSTOM["⚙️ Eigenes Template\nSiehe Aufgabe 2"]

    style A      fill:#4CAF50,color:#fff
    style A2     fill:#4CAF50,color:#fff
    style B      fill:#F44336,color:#fff
    style C      fill:#00BCD4,color:#000
    style CUSTOM fill:#FF9800,color:#000
    style Q1     fill:#37474F,color:#fff
    style Q2     fill:#37474F,color:#fff
    style Q3     fill:#37474F,color:#fff
    style Q4     fill:#37474F,color:#fff
'''
mermaid(diag_e, width=780)

# 4 | Supervisor + 2 Worker implementieren
---

Wir implementieren **Template A: Content Pipeline** vollständig.

<p><font color='black' size="5">Neu in M20: Prompts aus Datei laden</font></p>

Statt Supervisor- und Worker-Prompts inline zu schreiben, laden wir sie aus
Markdown-Dateien im `05_prompt/`-Verzeichnis. Das macht sie:
- ✅ Wiederverwendbar (verschiedene Notebooks, gleiche Prompts)
- ✅ Leicht editierbar (kein Notebook-Restart nötig)
- ✅ Versionierbar (Git-History für Prompt-Änderungen)

```python
from genai_lib.utilities import load_prompt

# mode="S" → reiner String  (für SystemMessage)
# mode="T" → ChatPromptTemplate  (Standard, für LCEL-Chains)
supervisor_system = load_prompt("../05_prompt/m20_supervisor_prompt.md", mode="S")
```

Die Prompt-Dateien verwenden `## system` als Abschnitts-Marker –
das Format, das `load_prompt()` in beiden Modi versteht.

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────
from typing import Annotated, Literal
from typing_extensions import TypedDict
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import create_react_agent
from IPython.display import Image as IPImage

# ── LLM ───────────────────────────────────────────────────────────────────
llm = init_chat_model("openai:gpt-4o-mini", temperature=0.0)
print("✅ LLM initialisiert")

In [ ]:
# ── Tool: Recherche ──────────────────────────────────────────────────────
import wikipedia as wiki_lib

@tool
def wikipedia_suche(suchbegriff: str) -> str:
    'Sucht Informationen in Wikipedia (5 Saetze). Englische Begriffe bevorzugen.'
    wiki_lib.set_lang('en')
    try:
        return wiki_lib.summary(suchbegriff, sentences=5)
    except wiki_lib.exceptions.DisambiguationError as e:
        try:
            return wiki_lib.summary(e.options[0], sentences=5)
        except Exception:
            return f'Mehrdeutig. Alternativen: {e.options[:3]}'
    except wiki_lib.exceptions.PageError:
        try:
            return wiki_lib.summary(f'{suchbegriff} programming', sentences=5)
        except Exception as e2:
            return f'Nicht gefunden: {suchbegriff}. {e2}'
    except Exception as e:
        return f'Fehler: {str(e)}'

# ── Tools: Schreiben ──────────────────────────────────────────────────────
@tool
def gliederung_erstellen(thema: str) -> str:
    'Erstellt eine strukturierte Gliederung fuer einen Artikel zum Thema.'
    return (
        f'Gliederung fuer "{thema}":\n'
        '1. Einleitung & Definition\n'
        '2. Hintergrund & Entstehung\n'
        '3. Aktuelle Bedeutung & Anwendungen\n'
        '4. Fazit & Ausblick'
    )

@tool
def wort_zaehlen(text: str) -> str:
    'Zaehlt Woerter in einem Text. Gibt Feedback ob das Wortlimit eingehalten wird.'
    anzahl = len(text.split())
    if anzahl < 150:
        hinweis = '⚠️ Text zu kurz – erweitern.'
    elif anzahl > 350:
        hinweis = '⚠️ Text zu lang – kuerzen.'
    else:
        hinweis = '✅ Wortanzahl optimal.'
    return f'Woerter: {anzahl}. {hinweis}'

print('✅ Tools: wikipedia_suche, gliederung_erstellen, wort_zaehlen')

<p><font color='black' size="5">Prompts aus Datei laden</font></p>

`load_prompt()` liest eine Markdown-Datei und gibt den Text als String zurück.
So können Prompts zentral gepflegt werden – ohne das Notebook neu zu starten.

In [ ]:
# ── Prompts aus 05_prompt/ laden ─────────────────────────────────────────
SUPERVISOR_SYSTEM = load_prompt("https://github.com/ralf-42/Agenten/blob/main/05_prompt/m20_supervisor_prompt.md", mode="S")
RECHERCHE_SYSTEM  = load_prompt("https://github.com/ralf-42/Agenten/blob/main/05_prompt/m20_recherche_prompt.md", mode="S")
SCHREIB_SYSTEM    = load_prompt("https://github.com/ralf-42/Agenten/blob/main/05_prompt/m20_schreib_prompt.md", mode="S")

print("✅ Prompts geladen:")
print(f"   Supervisor: {len(SUPERVISOR_SYSTEM)} Zeichen")
print(f"   Recherche:  {len(RECHERCHE_SYSTEM)} Zeichen")
print(f"   Schreiben:  {len(SCHREIB_SYSTEM)} Zeichen")

# ── Worker Agents ────────────────────────────────────────────────────────
INNER_CFG = {'recursion_limit': 15}

recherche_agent = create_react_agent(
    llm,
    tools=[wikipedia_suche],
    prompt=SystemMessage(RECHERCHE_SYSTEM),
)

schreib_agent = create_react_agent(
    llm,
    tools=[gliederung_erstellen, wort_zaehlen],
    prompt=SystemMessage(SCHREIB_SYSTEM),
)

print("\n✅ Worker Agents:")
print("   recherche_agent → [wikipedia_suche]")
print("   schreib_agent   → [gliederung_erstellen, wort_zaehlen]")

<p><font color='black' size="5">State und Supervisor</font></p>

Der State speichert alle Informationen die der Graph zwischen Schritten braucht:

```python
class ProjektState(TypedDict):
    messages:    Annotated[list, add_messages]  # Gesamt-History
    aufgabe:     str    # Original-Aufgabe (unveraenderlich)
    naechster:   str    # Routing-Entscheidung
    iterationen: int    # Wie viele Worker haben gearbeitet?
    max_iter:    int    # Obergrenze
```

`with_structured_output()` erzwingt eine strukturierte JSON-Ausgabe vom Supervisor –
kein Freitext-Parsing nötig, kein Halluzinations-Risiko beim Routing.

In [ ]:
# ── State ────────────────────────────────────────────────────────────────
class ProjektState(TypedDict):
    messages:    Annotated[list, add_messages]
    aufgabe:     str
    naechster:   str
    iterationen: int
    max_iter:    int

# ── Pydantic-Modell für strukturiertes Routing ────────────────────────────
class SupervisorEntscheidung(BaseModel):
    naechster: Literal['recherche', 'schreiben', 'FINISH'] = Field(
        description='Naechster Agent oder FINISH wenn abgeschlossen.'
    )
    begruendung: str = Field(
        description='Kurze Begruendung fuer diese Entscheidung (1 Satz).'
    )

supervisor_llm = llm.with_structured_output(SupervisorEntscheidung)

# ── Supervisor Node ───────────────────────────────────────────────────────
def supervisor_node(state: ProjektState) -> dict:
    if state['iterationen'] >= state['max_iter']:
        mprint(f"⚠️ **Iterations-Limit ({state['max_iter']}) erreicht – FINISH.**")
        return {'naechster': 'FINISH'}
    entscheidung = supervisor_llm.invoke([
        SystemMessage(SUPERVISOR_SYSTEM),
        HumanMessage(f"Aufgabe: {state['aufgabe']}"),
        *state['messages'],
    ])
    mprint(f"**👔 Supervisor →** `{entscheidung.naechster}`  _{entscheidung.begruendung}_")
    return {'naechster': entscheidung.naechster}

# ── Router ────────────────────────────────────────────────────────────────
def supervisor_router(state: ProjektState) -> str:
    naechster = state.get('naechster', 'FINISH')
    return '__end__' if naechster == 'FINISH' else naechster

print('✅ State, SupervisorEntscheidung, supervisor_node bereit')

In [ ]:
# ── Worker Node Factory ──────────────────────────────────────────────────
def make_worker_node(agent, name: str):
    def worker_node(state: ProjektState) -> dict:
        mprint(f'\n**🤖 {name}** arbeitet...')
        kontext = f"Aufgabe: {state['aufgabe']}"
        if state['messages']:
            bisherige = '\n'.join(
                f"{getattr(m, 'name', 'System')}: {m.content[:400]}"
                for m in state['messages']
            )
            kontext += f'\n\nBisherige Arbeit des Teams:\n{bisherige}'
        result = agent.invoke(
            {'messages': [HumanMessage(kontext)]},
            config=INNER_CFG,
        )
        letzte = result['messages'][-1]
        return {
            'messages':    [AIMessage(content=letzte.content, name=name)],
            'iterationen': state['iterationen'] + 1,
        }
    worker_node.__name__ = f'{name}_node'
    return worker_node

recherche_node = make_worker_node(recherche_agent, 'Recherche')
schreiben_node = make_worker_node(schreib_agent,   'Schreiben')
print('✅ Worker-Nodes erstellt')

In [ ]:
# ── COMPILE ──────────────────────────────────────────────────────────────
builder = StateGraph(ProjektState)

builder.add_node('supervisor', supervisor_node)
builder.add_node('recherche',  recherche_node)
builder.add_node('schreiben',  schreiben_node)

builder.add_edge(START, 'supervisor')

builder.add_conditional_edges(
    'supervisor',
    supervisor_router,
    {
        'recherche': 'recherche',
        'schreiben': 'schreiben',
        '__end__':   END,
    }
)

builder.add_edge('recherche', 'supervisor')
builder.add_edge('schreiben', 'supervisor')

projekt_graph = builder.compile()
print('✅ Projekt-Graph kompiliert (Template A: Content Pipeline)')

In [ ]:
# ── VIZ: LangGraph-eigene Visualisierung ─────────────────────────────────
try:
    display(IPImage(projekt_graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f'(Graph-Visualisierung nicht verfuegbar: {e})')

In [ ]:
# ── TEST 1: Thema Quantencomputing ──────────────────────────────────────
aufgabe1 = (
    'Schreibe einen strukturierten Artikel über Quantencomputing: '
    'Was ist es, wie funktioniert es, welche Bedeutung hat es für die Zukunft?'
)

result1 = projekt_graph.invoke(
    {
        'messages':    [],
        'aufgabe':     aufgabe1,
        'naechster':   '',
        'iterationen': 0,
        'max_iter':    4,
    },
    config={'run_name': 'M20-Test1-Quantencomputing', 'tags': ['m20'],
            'recursion_limit': 25}
)

mprint('## 🎯 Test 1: Quantencomputing-Artikel')
mprint(f"**Iterationen:** {result1['iterationen']} von max. 4\n")
mprint('---')
for msg in result1['messages']:
    agent = getattr(msg, 'name', None) or 'System'
    mprint(f'**[{agent}]**\n{msg.content}\n---')

In [ ]:
# ── TEST 2: Thema Machine Learning ──────────────────────────────────────
aufgabe2 = (
    'Erstelle einen Überblick über Machine Learning: '
    'Typen, Algorithmen und aktuelle Anwendungen in der Industrie.'
)

result2 = projekt_graph.invoke(
    {
        'messages':    [],
        'aufgabe':     aufgabe2,
        'naechster':   '',
        'iterationen': 0,
        'max_iter':    4,
    },
    config={'run_name': 'M20-Test2-MachineLearning', 'tags': ['m20'],
            'recursion_limit': 25}
)

mprint('## 🎯 Test 2: Machine-Learning-Überblick')
mprint(f"**Iterationen:** {result2['iterationen']}\n")
letzte = result2['messages'][-1]
mprint(f"**Finale Antwort [{getattr(letzte, 'name', '?')}]:**\n{letzte.content}")

# 5 | Debugging mit LangSmith
---

**LangSmith** protokolliert jeden Graph-Schritt und macht das Innenleben sichtbar.

**Was LangSmith zeigt:**
- Welcher Agent wann aufgerufen wurde
- Welche Tools mit welchen Argumenten benutzt wurden
- Token-Verbrauch pro Schritt
- Gesamt-Latenz und Kosten

**Konfiguration in diesem Notebook:**
```python
os.environ["LANGSMITH_PROJECT"] = "M20-Multi-Agent-Projekt"
os.environ["LANGSMITH_TRACING"] = "true"
```

Alle Runs erscheinen im LangSmith-Dashboard unter `M20-Multi-Agent-Projekt`.    
**Lokale Analyse** mit der Hilfsfunktion unten ergänzt das Tracing:

In [ ]:
# ── Lauf-Analyse Funktion ────────────────────────────────────────────────
def analysiere_lauf(result: dict, title: str = 'Lauf-Analyse'):
    messages = result['messages']
    beitraege: dict = {}
    zeichen:   dict = {}
    for msg in messages:
        name = getattr(msg, 'name', None) or 'Unbekannt'
        beitraege[name] = beitraege.get(name, 0) + 1
        zeichen[name]   = zeichen.get(name, 0) + len(msg.content)
    mprint(f'## 📊 {title}')
    mprint(f"**Iterationen:** {result.get('iterationen', '-')}")
    mprint(f'**Nachrichten gesamt:** {len(messages)}')
    mprint('\n**Agent-Beiträge:**')
    for agent in sorted(beitraege):
        mprint(f'  - `{agent}`: {beitraege[agent]}x | {zeichen[agent]} Zeichen')
    if messages:
        laengste = max(messages, key=lambda m: len(m.content))
        mprint(f"\n**Längste Nachricht:** `{getattr(laengste, 'name', '?')}` "
               f'({len(laengste.content)} Zeichen)')

# ── Analyse beider Test-Läufe ─────────────────────────────────────────────
analysiere_lauf(result1, 'Test 1: Quantencomputing')
print()
analysiere_lauf(result2, 'Test 2: Machine Learning')

# 6 | MVP-Definition
---

Ein **MVP (Minimum Viable Product)** für ein Multi-Agent-System ist lauffähig,
getestet und klar dokumentiert – aber noch nicht production-ready.



<p><font color='black' size="5">MVP-Checkliste</font></p>

| Kriterium | Beschreibung | Status in M20 |
|-----------|-------------|---------------|
| ✅ **Läuft durch** | Graph endet ohne Fehler | Getestet |
| ✅ **Iterations-Schutz** | `max_iter` + `recursion_limit` | Implementiert |
| ✅ **Strukturiertes Routing** | `with_structured_output` | Implementiert |
| ✅ **Prompts extern** | `load_prompt()` statt Inline | Implementiert |
| ✅ **LangSmith-Tracing** | `run_name` + `tags` | Konfiguriert |
| ⚠️ **Fehlerbehandlung** | Try/Except in Worker-Nodes | Fehlt noch |
| ⚠️ **Checkpointing** | SQLite-Persistenz | Fehlt noch |
| ⚠️ **Tests** | Unit-Tests für Tools | Fehlt noch |
| ❌ **UI** | Gradio / Streamlit | Nicht in MVP |



<p><font color='black' size="5">Was kommt nach dem MVP?</font></p>

- **M20 (OpenAI Agent Builder):** Alternativer Ansatz ohne LangGraph-Boilerplate – kurze Demo als Ausblick
- **M22 (Agentic RAG):** RAG-Agenten mit adaptivem Retrieval und Multi-Hop-Reasoning
- **M26 (Gradio UI für Agenten):** Web-Interface direkt für den eigenen Agenten
- **M28 (Production Deployment):** Fehlerbehandlung, Tests, Docker, Monitoring

> 💡 **Tipp:** Starte immer mit dem MVP. Es ist besser, ein funktionierendes
> System mit 2 Agents zu haben als ein perfektes System das nie fertig wird.

# A | Aufgabe
---

Die Aufgabenstellungen bieten Anregungen. Du kannst auch eine andere Herausforderung angehen.

<p><font color='black' size="5">
Aufgabe 1: Template B implementieren – Analyse Pipeline
</font></p>

Implementiere **Template B: Analyse Pipeline** mit zwei Worker Agents:

- **Erfassungs-Agent**: Nimmt Daten entgegen und bereitet sie auf
- **Analyse-Agent**: Wertet aus und erstellt einen strukturierten Bericht

Nutze diese Tools:
```python
@tool
def python_ausfuehren(code: str) -> str:
    'Fuehrt Python-Code aus und gibt das Ergebnis zurueck.'
    ...

@tool
def kennzahlen_berechnen(zahlen: str) -> str:
    'Berechnet Min, Max, Durchschnitt einer kommaseparierten Zahlenliste.'
    ...
```

**Test-Aufgabe:**  
*"Analysiere die Zahlenreihe [4, 7, 2, 9, 1, 5, 8, 3, 6] und
schreibe einen Bericht über Verteilung und Auffälligkeiten."*

<p><font color='black' size="5">
Aufgabe 2: Eigenes Thema mit Template A
</font></p>

Wähle ein eigenes Thema und erstelle damit einen Artikel mit Template A.

**Vorschläge:**
- Blockchain-Technologie und ihre Anwendungen
- Reinforcement Learning – Grundlagen und Beispiele
- Natural Language Processing: Von Regex bis Transformer

**Erweitere optional:**
Füge einen dritten Worker Agent hinzu – einen **Lektorat-Agent** der Stil und
Grammatik prüft. Lege dafür einen neuen Prompt in `05_prompt/` an.

<p><font color='black' size="5">
Aufgabe 3: Fehlertoleranz einbauen
</font></p>

Ersetze `make_worker_node()` durch eine fehlertolerante Version:

```python
def make_resilient_worker(agent, name: str, max_retries: int = 1):
    def resilient_node(state: ProjektState) -> dict:
        for versuch in range(1, max_retries + 2):
            try:
                result = agent.invoke(
                    {'messages': [HumanMessage(kontext)]},
                    config=INNER_CFG,
                )
                return {'messages': [...], 'iterationen': state['iterationen'] + 1}
            except Exception as e:
                if versuch <= max_retries:
                    mprint(f'⚠️ Versuch {versuch}: {e}. Wiederhole...')
                else:
                    return {
                        'messages': [AIMessage(
                            content=f'❌ {name}: {e}', name='System'
                        )],
                        'iterationen': state['iterationen'] + 1,
                    }
    return resilient_node
```

**Test:** Simuliere einen Fehler, indem `wikipedia_suche` bei einem bestimmten
Begriff absichtlich eine Exception wirft. Prüfe in LangSmith ob der Supervisor
korrekt reagiert und FINISH auslöst.